# 04. Evaluation разметки и предсказаний

Этот ноутбук сравнивает **любые два CSV-файла** в единой annotation-схеме:

```text
news_id
published_at
topic
title
text
cluster_id
novelty_label
comment
needs_review
```

Файлы могут быть любыми источниками:

- human vs LLM;
- human vs baseline;
- human vs model;
- LLM vs model;
- silver LLM vs model.

Первый файл считается **reference**, второй — **candidate**.

Важно: значения `cluster_id` между файлами могут называться по-разному. Поэтому качество группировки считается не через прямое равенство ID, а через **pairwise clustering metrics**: две новости должны быть в одном кластере или в разных кластерах.

## 1. Imports & configuration

In [1]:
from pathlib import Path
import json
from math import comb

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

In [ ]:
# Поменяй эти пути под нужное сравнение.
# Например:
# REFERENCE_PATH = DATA_DIR / "lenta_golden_set_human.csv"
# CANDIDATE_PATH = DATA_DIR / "lenta_golden_set_llm.csv"
# CANDIDATE_PATH = DATA_DIR / "lenta_baseline_predictions.csv"
# CANDIDATE_PATH = DATA_DIR / "lenta_model_predictions.csv"

# Основные пути
PROJECT_ROOT = Path.cwd().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PREPARED_DIR = DATA_DIR / "prepared"
OUTPUT_DIR = Path("./evaluation_reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_PATH = PREPARED_DIR / "lenta_golden_set_human.csv"
CANDIDATE_PATH = PREPARED_DIR / "lenta_golden_set_llm.csv"

REFERENCE_NAME = "reference"
CANDIDATE_NAME = "candidate"

SCHEMA_COLUMNS = [
    "news_id",
    "published_at",
    "topic",
    "title",
    "text",
    "cluster_id",
    "novelty_label",
    "comment",
    "needs_review",
]

KEY_COLUMN = "news_id"
CLUSTER_COLUMN = "cluster_id"
LABEL_COLUMN = "novelty_label"
NEEDS_REVIEW_COLUMN = "needs_review"

VALID_LABELS = [
    "significant",
    "minor",
    "duplicate",
    "wrong_cluster",
    "unclear",
]

IMPORTANT_POSITIVE_LABEL = "significant"
IMPORTANT_NEGATIVE_LABELS = ["minor", "duplicate"]
EXCLUDED_LABELS = ["wrong_cluster", "unclear", ""]

# Целевые пороги из MVP-бенчмарков.
BENCHMARK_TARGETS = {
    "coverage": (0.95, ">="),
    "pairwise_f1": (0.75, ">="),
    "false_merge_rate": (0.15, "<="),
    "important_update_f1": (0.75, ">="),
    "important_update_recall": (0.80, ">="),
}

## 2. Helper functions

In [3]:
def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Не найден файл: {path}\n"
            f"Ожидался: {description}.\n"
            "Проверь пути в конфигурационной ячейке."
        )


def normalize_bool_series(series: pd.Series) -> pd.Series:
    """Normalize bool-like values to pandas BooleanDtype."""
    normalized = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )
    true_values = {"true", "1", "yes", "y", "да", "истина"}
    false_values = {"false", "0", "no", "n", "нет", "ложь", ""}

    def parse_bool(value: str):
        if value in true_values:
            return True
        if value in false_values:
            return False
        return pd.NA

    return normalized.map(parse_bool).astype("boolean")


def normalize_annotation_df(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    """Validate and normalize annotation/prediction file schema."""
    missing = [col for col in SCHEMA_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"В {source_name} не хватает колонок: {missing}")

    result = df[SCHEMA_COLUMNS].copy()

    result[KEY_COLUMN] = result[KEY_COLUMN].astype(str).str.strip()
    result[CLUSTER_COLUMN] = result[CLUSTER_COLUMN].fillna("").astype(str).str.strip()
    result[LABEL_COLUMN] = (
        result[LABEL_COLUMN]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )
    result["topic"] = result["topic"].fillna("").astype(str).str.strip()
    result["title"] = result["title"].fillna("").astype(str)
    result["text"] = result["text"].fillna("").astype(str)
    result["comment"] = result["comment"].fillna("").astype(str)
    result[NEEDS_REVIEW_COLUMN] = normalize_bool_series(result[NEEDS_REVIEW_COLUMN])
    result["published_at"] = pd.to_datetime(result["published_at"], errors="coerce")

    duplicated = result[result.duplicated(KEY_COLUMN, keep=False)]
    if not duplicated.empty:
        display(duplicated.sort_values(KEY_COLUMN).head(20))
        raise ValueError(f"В {source_name} есть дубли по {KEY_COLUMN}. Сравнение по news_id станет неоднозначным.")

    unknown_labels = sorted(set(result[LABEL_COLUMN]) - set(VALID_LABELS) - {""})
    if unknown_labels:
        print(f"WARNING: В {source_name} есть неизвестные novelty_label: {unknown_labels}")

    empty_clusters = result[result[CLUSTER_COLUMN].eq("")]
    if not empty_clusters.empty:
        print(f"WARNING: В {source_name} есть строки с пустым cluster_id: {len(empty_clusters)}")

    return result


def load_annotation_file(path: Path, source_name: str) -> pd.DataFrame:
    require_file(path, source_name)
    df = pd.read_csv(path)
    return normalize_annotation_df(df, source_name)

In [4]:
def comb2(n: int) -> int:
    return n * (n - 1) // 2 if n >= 2 else 0


def compute_pairwise_clustering_metrics(
    reference_cluster_ids: pd.Series,
    candidate_cluster_ids: pd.Series,
) -> dict:
    """
    Compare two cluster assignments without requiring cluster_id values to match.

    Pair-level interpretation:
    TP: pair is same cluster in both reference and candidate
    FP: pair is same cluster in candidate, but different in reference = false merge
    FN: pair is different in candidate, but same in reference = false split
    TN: pair is different in both
    """
    ref = reference_cluster_ids.fillna("").astype(str)
    cand = candidate_cluster_ids.fillna("").astype(str)

    if len(ref) != len(cand):
        raise ValueError("reference_cluster_ids and candidate_cluster_ids must have equal length")

    n = len(ref)
    total_pairs = comb2(n)

    contingency = pd.crosstab(ref, cand)

    tp = int(sum(comb2(int(value)) for value in contingency.to_numpy().ravel()))
    ref_same_pairs = int(sum(comb2(int(value)) for value in contingency.sum(axis=1).to_numpy()))
    cand_same_pairs = int(sum(comb2(int(value)) for value in contingency.sum(axis=0).to_numpy()))

    fp = cand_same_pairs - tp
    fn = ref_same_pairs - tp
    tn = total_pairs - tp - fp - fn

    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else np.nan

    false_merge_rate = fp / (tp + fp) if (tp + fp) else np.nan
    false_split_rate = fn / (tp + fn) if (tp + fn) else np.nan

    return {
        "n_items": n,
        "total_pairs": total_pairs,
        "tp_same_pairs": tp,
        "fp_false_merge_pairs": fp,
        "fn_false_split_pairs": fn,
        "tn_different_pairs": tn,
        "reference_same_pairs": ref_same_pairs,
        "candidate_same_pairs": cand_same_pairs,
        "pairwise_precision": precision,
        "pairwise_recall": recall,
        "pairwise_f1": f1,
        "false_merge_rate": false_merge_rate,
        "false_split_rate": false_split_rate,
        "adjusted_rand_index": adjusted_rand_score(ref, cand),
        "normalized_mutual_info": normalized_mutual_info_score(ref, cand),
    }


def binary_important_label(label: str):
    """Map novelty labels to binary important-update target."""
    if label == IMPORTANT_POSITIVE_LABEL:
        return 1
    if label in IMPORTANT_NEGATIVE_LABELS:
        return 0
    return np.nan


def status_against_target(metric: str, value: float) -> str:
    if metric not in BENCHMARK_TARGETS or pd.isna(value):
        return "INFO"

    target, direction = BENCHMARK_TARGETS[metric]
    if direction == ">=":
        return "OK" if value >= target else "WARN"
    if direction == "<=":
        return "OK" if value <= target else "WARN"
    return "INFO"


def target_as_text(metric: str) -> str:
    if metric not in BENCHMARK_TARGETS:
        return ""
    target, direction = BENCHMARK_TARGETS[metric]
    return f"{direction} {target:g}"

## 3. Load files

Первый файл считается эталоном (`reference`), второй — сравниваемым результатом (`candidate`).

In [5]:
reference_df = load_annotation_file(REFERENCE_PATH, REFERENCE_NAME)
candidate_df = load_annotation_file(CANDIDATE_PATH, CANDIDATE_NAME)

print(f"{REFERENCE_NAME}:", reference_df.shape, REFERENCE_PATH)
print(f"{CANDIDATE_NAME}:", candidate_df.shape, CANDIDATE_PATH)

reference_df.head(3)

FileNotFoundError: Не найден файл: lenta_golden_set_human.csv
Ожидался: reference.
Проверь пути в конфигурационной ячейке.

In [ ]:
candidate_df.head(3)

## 4. Coverage and merge by `news_id`

Сопоставление файлов выполняется по `news_id`.

`cluster_id` не используется как ключ, потому что разные источники могут по-разному назвать одни и те же кластеры.

In [ ]:
reference_ids = set(reference_df[KEY_COLUMN])
candidate_ids = set(candidate_df[KEY_COLUMN])

only_reference_ids = sorted(reference_ids - candidate_ids)
only_candidate_ids = sorted(candidate_ids - reference_ids)
common_ids = sorted(reference_ids & candidate_ids)

coverage = len(common_ids) / len(reference_ids) if reference_ids else np.nan

coverage_report = pd.DataFrame([
    {"metric": "reference_rows", "value": len(reference_df)},
    {"metric": "candidate_rows", "value": len(candidate_df)},
    {"metric": "common_news_id", "value": len(common_ids)},
    {"metric": "only_reference", "value": len(only_reference_ids)},
    {"metric": "only_candidate", "value": len(only_candidate_ids)},
    {"metric": "coverage", "value": coverage},
])

coverage_report

In [ ]:
merged = reference_df.merge(
    candidate_df,
    on=KEY_COLUMN,
    how="inner",
    suffixes=(f"_{REFERENCE_NAME}", f"_{CANDIDATE_NAME}"),
)

print("merged:", merged.shape)
merged.head(3)

## 5. Label distribution and sanity checks

In [ ]:
def label_distribution(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    counts = (
        df[LABEL_COLUMN]
        .replace("", "<empty/story_start>")
        .value_counts(dropna=False)
        .rename_axis("novelty_label")
        .reset_index(name="count")
    )
    counts["share"] = counts["count"] / len(df) if len(df) else np.nan
    counts["source"] = source_name
    return counts[["source", "novelty_label", "count", "share"]]

label_distribution_report = pd.concat([
    label_distribution(reference_df, REFERENCE_NAME),
    label_distribution(candidate_df, CANDIDATE_NAME),
], ignore_index=True)

label_distribution_report

In [ ]:
needs_review_report = pd.DataFrame([
    {
        "source": REFERENCE_NAME,
        "rows": len(reference_df),
        "needs_review_count": int(reference_df[NEEDS_REVIEW_COLUMN].fillna(False).sum()),
        "needs_review_rate": float(reference_df[NEEDS_REVIEW_COLUMN].fillna(False).mean()),
    },
    {
        "source": CANDIDATE_NAME,
        "rows": len(candidate_df),
        "needs_review_count": int(candidate_df[NEEDS_REVIEW_COLUMN].fillna(False).sum()),
        "needs_review_rate": float(candidate_df[NEEDS_REVIEW_COLUMN].fillna(False).mean()),
    },
])

needs_review_report

## 6. Multi-class novelty comparison

Пустая `novelty_label` трактуется как старт сюжета и не участвует в сравнении novelty-update.

В multiclass-сравнение входят только строки, где у обоих источников есть непустая метка из допустимого списка.

In [ ]:
ref_label_col = f"{LABEL_COLUMN}_{REFERENCE_NAME}"
cand_label_col = f"{LABEL_COLUMN}_{CANDIDATE_NAME}"

multiclass_mask = (
    merged[ref_label_col].isin(VALID_LABELS)
    & merged[cand_label_col].isin(VALID_LABELS)
)

multiclass_df = merged[multiclass_mask].copy()

print("Rows with comparable novelty labels:", len(multiclass_df), "of", len(merged))

if len(multiclass_df) == 0:
    print("No comparable novelty labels. Fill novelty_label columns first.")
else:
    y_ref = multiclass_df[ref_label_col]
    y_cand = multiclass_df[cand_label_col]

    multiclass_metrics = {
        "novelty_accuracy": accuracy_score(y_ref, y_cand),
        "novelty_macro_f1": f1_score(y_ref, y_cand, labels=VALID_LABELS, average="macro", zero_division=0),
        "novelty_weighted_f1": f1_score(y_ref, y_cand, labels=VALID_LABELS, average="weighted", zero_division=0),
        "cohen_kappa": cohen_kappa_score(y_ref, y_cand),
        "novelty_comparable_rows": len(multiclass_df),
    }

    display(pd.DataFrame([multiclass_metrics]).T.rename(columns={0: "value"}))

In [ ]:
if len(multiclass_df) > 0:
    cm = pd.DataFrame(
        confusion_matrix(y_ref, y_cand, labels=VALID_LABELS),
        index=[f"ref:{label}" for label in VALID_LABELS],
        columns=[f"cand:{label}" for label in VALID_LABELS],
    )
    display(cm)

    print(classification_report(
        y_ref,
        y_cand,
        labels=VALID_LABELS,
        zero_division=0,
    ))
else:
    cm = pd.DataFrame()
    multiclass_metrics = {
        "novelty_accuracy": np.nan,
        "novelty_macro_f1": np.nan,
        "novelty_weighted_f1": np.nan,
        "cohen_kappa": np.nan,
        "novelty_comparable_rows": 0,
    }

## 7. Important update metrics

Для MVP важнее всего не пропускать важные обновления.

Бинаризация:

```text
significant -> 1
minor       -> 0
duplicate   -> 0
```

Исключаются:

```text
wrong_cluster
unclear
пустая novelty_label / старт сюжета
```

In [ ]:
important_df = merged.copy()
important_df["important_ref"] = important_df[ref_label_col].map(binary_important_label)
important_df["important_cand"] = important_df[cand_label_col].map(binary_important_label)

important_eval_df = important_df.dropna(subset=["important_ref", "important_cand"]).copy()
important_eval_df["important_ref"] = important_eval_df["important_ref"].astype(int)
important_eval_df["important_cand"] = important_eval_df["important_cand"].astype(int)

print("Rows for important-update binary evaluation:", len(important_eval_df), "of", len(merged))

if len(important_eval_df) == 0:
    important_metrics = {
        "important_update_precision": np.nan,
        "important_update_recall": np.nan,
        "important_update_f1": np.nan,
        "important_update_support": 0,
        "important_update_positive_support": 0,
    }
else:
    y_ref_imp = important_eval_df["important_ref"]
    y_cand_imp = important_eval_df["important_cand"]

    important_metrics = {
        "important_update_precision": precision_score(y_ref_imp, y_cand_imp, zero_division=0),
        "important_update_recall": recall_score(y_ref_imp, y_cand_imp, zero_division=0),
        "important_update_f1": f1_score(y_ref_imp, y_cand_imp, zero_division=0),
        "important_update_support": len(important_eval_df),
        "important_update_positive_support": int(y_ref_imp.sum()),
    }

pd.DataFrame([important_metrics]).T.rename(columns={0: "value"})

In [ ]:
if len(important_eval_df) > 0:
    important_cm = pd.DataFrame(
        confusion_matrix(y_ref_imp, y_cand_imp, labels=[0, 1]),
        index=["ref:not_significant", "ref:significant"],
        columns=["cand:not_significant", "cand:significant"],
    )
    display(important_cm)
else:
    important_cm = pd.DataFrame()

## 8. Pairwise clustering metrics

`cluster_id` может называться по-разному в двух файлах. Поэтому сравниваем не сами ID, а структуру группировки.

Для каждой пары новостей проверяем:

- в одном ли кластере они у reference;
- в одном ли кластере они у candidate.

Основные метрики:

- `pairwise_f1` — качество группировки инфоповодов;
- `false_merge_rate` — доля ошибочных склеек среди пар, которые candidate склеил;
- `false_split_rate` — доля ошибочных разделений среди пар, которые reference считает одним сюжетом.

In [ ]:
ref_cluster_col = f"{CLUSTER_COLUMN}_{REFERENCE_NAME}"
cand_cluster_col = f"{CLUSTER_COLUMN}_{CANDIDATE_NAME}"

cluster_metrics = compute_pairwise_clustering_metrics(
    merged[ref_cluster_col],
    merged[cand_cluster_col],
)

pd.DataFrame([cluster_metrics]).T.rename(columns={0: "value"})

## 9. Error analysis

In [ ]:
novelty_disagreements = multiclass_df[
    multiclass_df[ref_label_col] != multiclass_df[cand_label_col]
].copy() if len(multiclass_df) > 0 else pd.DataFrame()

print("Novelty disagreements:", len(novelty_disagreements))

if len(novelty_disagreements) > 0:
    display_columns = [
        KEY_COLUMN,
        f"published_at_{REFERENCE_NAME}",
        f"topic_{REFERENCE_NAME}",
        f"title_{REFERENCE_NAME}",
        ref_label_col,
        cand_label_col,
        f"comment_{REFERENCE_NAME}",
        f"comment_{CANDIDATE_NAME}",
    ]
    display(novelty_disagreements[display_columns].head(20))

In [ ]:
if len(important_eval_df) > 0:
    important_false_negatives = important_eval_df[
        (important_eval_df["important_ref"] == 1)
        & (important_eval_df["important_cand"] == 0)
    ].copy()

    important_false_positives = important_eval_df[
        (important_eval_df["important_ref"] == 0)
        & (important_eval_df["important_cand"] == 1)
    ].copy()
else:
    important_false_negatives = pd.DataFrame()
    important_false_positives = pd.DataFrame()

print("Important false negatives:", len(important_false_negatives))
print("Important false positives:", len(important_false_positives))

if len(important_false_negatives) > 0:
    display(important_false_negatives[[
        KEY_COLUMN,
        f"published_at_{REFERENCE_NAME}",
        f"topic_{REFERENCE_NAME}",
        f"title_{REFERENCE_NAME}",
        ref_label_col,
        cand_label_col,
        f"comment_{REFERENCE_NAME}",
        f"comment_{CANDIDATE_NAME}",
    ]].head(20))

In [ ]:
def sample_pairwise_cluster_errors(
    df: pd.DataFrame,
    ref_cluster_col: str,
    cand_cluster_col: str,
    max_pairs: int = 30,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Sample false merge and false split pairs for manual inspection."""
    # This is O(n^2), so use it only for inspection-sized data.
    rng = np.random.default_rng(random_state)
    rows = df[[KEY_COLUMN, ref_cluster_col, cand_cluster_col, f"title_{REFERENCE_NAME}"]].reset_index(drop=True)

    false_merges = []
    false_splits = []

    n = len(rows)
    for i in range(n):
        for j in range(i + 1, n):
            same_ref = rows.at[i, ref_cluster_col] == rows.at[j, ref_cluster_col]
            same_cand = rows.at[i, cand_cluster_col] == rows.at[j, cand_cluster_col]

            if same_cand and not same_ref:
                false_merges.append((i, j))
            elif same_ref and not same_cand:
                false_splits.append((i, j))

    def build_sample(pairs, error_type):
        if not pairs:
            return pd.DataFrame()
        selected_idx = rng.choice(len(pairs), size=min(max_pairs, len(pairs)), replace=False)
        records = []
        for idx in selected_idx:
            i, j = pairs[int(idx)]
            records.append({
                "error_type": error_type,
                "news_id_1": rows.at[i, KEY_COLUMN],
                "news_id_2": rows.at[j, KEY_COLUMN],
                "ref_cluster_1": rows.at[i, ref_cluster_col],
                "ref_cluster_2": rows.at[j, ref_cluster_col],
                "cand_cluster_1": rows.at[i, cand_cluster_col],
                "cand_cluster_2": rows.at[j, cand_cluster_col],
                "title_1": rows.at[i, f"title_{REFERENCE_NAME}"],
                "title_2": rows.at[j, f"title_{REFERENCE_NAME}"],
            })
        return pd.DataFrame(records)

    return build_sample(false_merges, "false_merge"), build_sample(false_splits, "false_split")


if len(merged) <= 3000:
    false_merge_sample, false_split_sample = sample_pairwise_cluster_errors(
        merged,
        ref_cluster_col,
        cand_cluster_col,
        max_pairs=30,
    )
    print("False merge sample:", len(false_merge_sample))
    print("False split sample:", len(false_split_sample))
else:
    false_merge_sample = pd.DataFrame()
    false_split_sample = pd.DataFrame()
    print("Pairwise error samples skipped: too many rows for O(n^2) inspection.")

if len(false_merge_sample) > 0:
    display(false_merge_sample.head(10))

## 10. Benchmark summary

Эта таблица выводит метрики, которые сейчас применимы к golden/silver/model CSV.

Метрики Top-10 deduplication и semantic search здесь намеренно не считаются: для них нужен отдельный формат ранжированной выдачи, которого нет в текущем golden set.

In [ ]:
summary_values = {
    "coverage": coverage,
    "pairwise_f1": cluster_metrics["pairwise_f1"],
    "false_merge_rate": cluster_metrics["false_merge_rate"],
    "false_split_rate": cluster_metrics["false_split_rate"],
    "important_update_precision": important_metrics["important_update_precision"],
    "important_update_recall": important_metrics["important_update_recall"],
    "important_update_f1": important_metrics["important_update_f1"],
    "novelty_accuracy": multiclass_metrics["novelty_accuracy"],
    "novelty_macro_f1": multiclass_metrics["novelty_macro_f1"],
    "novelty_weighted_f1": multiclass_metrics["novelty_weighted_f1"],
    "cohen_kappa": multiclass_metrics["cohen_kappa"],
    "reference_needs_review_rate": float(reference_df[NEEDS_REVIEW_COLUMN].fillna(False).mean()),
    "candidate_needs_review_rate": float(candidate_df[NEEDS_REVIEW_COLUMN].fillna(False).mean()),
}

benchmark_summary = pd.DataFrame([
    {
        "metric": metric,
        "value": value,
        "target": target_as_text(metric),
        "status": status_against_target(metric, value),
    }
    for metric, value in summary_values.items()
])

benchmark_summary

## 11. Save reports

In [ ]:
comparison_prefix = f"{REFERENCE_NAME}_vs_{CANDIDATE_NAME}"

coverage_report.to_csv(OUTPUT_DIR / f"{comparison_prefix}_coverage_report.csv", index=False)
label_distribution_report.to_csv(OUTPUT_DIR / f"{comparison_prefix}_label_distribution.csv", index=False)
needs_review_report.to_csv(OUTPUT_DIR / f"{comparison_prefix}_needs_review_report.csv", index=False)
benchmark_summary.to_csv(OUTPUT_DIR / f"{comparison_prefix}_benchmark_summary.csv", index=False)
merged.to_csv(OUTPUT_DIR / f"{comparison_prefix}_merged_comparison.csv", index=False)

if len(cm) > 0:
    cm.to_csv(OUTPUT_DIR / f"{comparison_prefix}_novelty_confusion_matrix.csv")

if len(important_cm) > 0:
    important_cm.to_csv(OUTPUT_DIR / f"{comparison_prefix}_important_update_confusion_matrix.csv")

if len(novelty_disagreements) > 0:
    novelty_disagreements.to_csv(OUTPUT_DIR / f"{comparison_prefix}_novelty_disagreements.csv", index=False)

if len(important_false_negatives) > 0:
    important_false_negatives.to_csv(OUTPUT_DIR / f"{comparison_prefix}_important_false_negatives.csv", index=False)

if len(important_false_positives) > 0:
    important_false_positives.to_csv(OUTPUT_DIR / f"{comparison_prefix}_important_false_positives.csv", index=False)

if len(false_merge_sample) > 0:
    false_merge_sample.to_csv(OUTPUT_DIR / f"{comparison_prefix}_false_merge_sample.csv", index=False)

if len(false_split_sample) > 0:
    false_split_sample.to_csv(OUTPUT_DIR / f"{comparison_prefix}_false_split_sample.csv", index=False)

summary_json = {
    "reference_name": REFERENCE_NAME,
    "candidate_name": CANDIDATE_NAME,
    "reference_path": str(REFERENCE_PATH),
    "candidate_path": str(CANDIDATE_PATH),
    "coverage": coverage,
    "multiclass_metrics": multiclass_metrics,
    "important_metrics": important_metrics,
    "cluster_metrics": cluster_metrics,
    "summary_values": summary_values,
}

with open(OUTPUT_DIR / f"{comparison_prefix}_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)

print("Reports saved to:", OUTPUT_DIR.resolve())

## 12. Interpretation notes

Как читать результаты:

- `pairwise_f1` — основная метрика качества группировки сюжетов. Она не зависит от совпадения имён `cluster_id`.
- `false_merge_rate` — насколько часто candidate ошибочно склеивает разные сюжеты. Это одна из самых важных метрик для кластеризации.
- `important_update_recall` — насколько candidate не пропускает важные обновления. Для продукта это особенно важно.
- `important_update_f1` — общий баланс precision/recall по важным обновлениям.
- `novelty_macro_f1` и `cohen_kappa` — дополнительные sanity-check метрики по много-классовой novelty-разметке.

Пустая `novelty_label` считается стартом сюжета и исключается из novelty-update метрик.

`wrong_cluster` и `unclear` также исключаются из бинарных important-update метрик, потому что это не обычные классы “важно / не важно”.